In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

**Q1 While ingesting customer data from an external source, you notice duplicate entries. How would you remove duplicates and retain only the latest entry based on a timestamp column?**

In [0]:
data = [
    ('101','2023-12-01',100),
    ('101','2023-12-02',200),
    ('102','2023-12-01',300),
    ('103','2023-12-02',400)

]
column = 'product_id','date','sales'

df = spark.createDataFrame(data,column)
df.display()

In [0]:
df.withColumn('date',col('date').cast(DateType()))

df = df.orderBy('product_id','date',ascending=[0,1]).dropDuplicates(subset=["product_id"])


In [0]:
df.display()

**2. While processing data from multiple files with inconsistent schemas, you need to merge them into a single DataFrame. How would you handle this inconsistency in PySpark?**

In [0]:
df = spark.read \
    .format("delta") \
    .option("mergeSchema", "true")\
    .load("uber.bronze.bulk_rides")


%md
**4. You are working with a real-time data pipeline, and you notice missing values in your streaming data Column - Category. How would you handle null or missing values in such a scenario?**

**df_stream = spark.readStream.schema("id INT, value STRING").csv("path/to/stream")**

In [0]:
df.fillna({'category':'N/A'})

**5. You need to calculate the total number of actions performed by users in a system. How would you calculate the top 5 most active users based on this information?**

In [0]:
data = [
    ('user1',4),
    ('user2',6),
    ('user3',9),
    ('user4',10),
    ('user3',2),
    ('user1',9),
    ('user5',2),
    ('user6',3)
]
column = 'user_id','actions'
df = spark.createDataFrame(data,column)
df.display()

In [0]:
df.groupBy('user_id').agg(sum('actions').alias('total_actions')).orderBy('total_actions',ascending=False).limit(5).display()


**6. While processing sales transaction data, you need to identify the most recent transaction for each customer. How would you approach this task?**

In [0]:
data = [("cust1", "2023-12-01", 100), ("cust2", "2023-12-02", 150),
        ("cust1", "2023-12-03", 200), ("cust2", "2023-12-04", 250)]
columns = ["customer_id", "transaction_date", "sales"]
df = spark.createDataFrame(data, columns)
df.display()

In [0]:
from pyspark.sql.window import *

In [0]:
df.withColumn('flag',dense_rank().over(Window.partitionBy('customer_id').orderBy(col('transaction_date').desc()))).filter(col("flag")==1).display()

**7. You need to identify customers who haven’t made any purchases in the last 30 days. How would you filter such customers?**

In [0]:
data = [("cust1", "2026-04-01"), ("cust2", "2025-12-20"), ("cust3", "2026-05-02")]
columns = ["customer_id", "last_purchase_date"]

df = spark.createDataFrame(data, columns)

df.display()

In [0]:
df.withColumn('gap',datediff(current_date(),col('last_purchase_date'))).filter(col('gap')>30).display()

**8. While analyzing customer reviews, you need to identify the most frequently used words in the feedback. How would you implement this?**

In [0]:
data = [("customer1", "The product is great"), ("customer2", "Great product, fast delivery"), ("customer3", "Not bad, could be better")]
columns = ["customer_id", "feedback"]

df = spark.createDataFrame(data, columns)

df.display()

In [0]:
df = df.withColumn('feedback',lower('feedback')).withColumn('feedback',explode(split('feedback',' ')))
df_grp = df.groupBy('feedback').agg(count('feedback').alias('wordCount'))
df_res = df_grp.orderBy(col('wordCount').desc())
df_res.display()

**9. You need to calculate the cumulative sum of sales over time for each product. How would you approach this?**

In [0]:
data = [("product1", "2023-12-01", 100), ("product2", "2023-12-02", 200),
        ("product1", "2023-12-03", 150), ("product2", "2023-12-04", 250)]
columns = ["product_id", "date", "sales"]
df = spark.createDataFrame(data, columns)
df.display()

In [0]:
df = df.withColumn('date',to_date(col('date')))
df = df.withColumn('cumSum',sum('sales').over(Window.partitionBy('product_id').orderBy(col('date'))))
df.display()

**10. While preparing a data pipeline, you notice some duplicate rows in a dataset. How would you remove the duplicates without affecting the original order?**

In [0]:
data = [("John", 25), ("Jane", 30), ("John", 25), ("Alice", 22)]
columns = ["name", "age"]
df = spark.createDataFrame(data, columns)
df.display()

In [0]:
df = df.withColumn('row_flag',row_number().over(Window.partitionBy('name').orderBy('age'))).filter(col('row_flag')==1)

df.display()

**11. You are working with user activity data and need to calculate the average session duration per user. How would you implement this?**

In [0]:
data = [("user1", "2023-12-01", 50), ("user1", "2023-12-02", 60), 
        ("user2", "2023-12-01", 45), ("user2", "2023-12-03", 75)]
columns = ["user_id", "session_date", "duration"]
df = spark.createDataFrame(data, columns)

df.display()

In [0]:
df.groupBy('user_id').agg(avg('duration').alias('avg_duration')).display()

**12. While analyzing sales data, you need to find the product with the highest sales for each month. How would you accomplish this?**

In [0]:
data = [("product1", "2023-12-01", 100), ("product2", "2023-12-01", 150), 
        ("product1", "2023-12-02", 200), ("product2", "2023-12-02", 250)]
columns = ["product_id", "date", "sales"]
df = spark.createDataFrame(data, columns)
df.display()

In [0]:
df.withColumn('date',to_date('date'))
df = df.withColumn('date',month('date')).groupBy('date','product_id').agg(sum('sales'))
df.display()